# Pseudolabeled data EDA

<a href="https://www.kaggle.com/code/gperdrizet/rsna-knee-abnormality-pseudolabel-eda?scriptVersionId=354735703" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>
<br>

## 1. Notebook setup
### 1.1. Imports

In [1]:
# Standard library
from pathlib import Path

# Third party
import pandas as pd

### 1.2. Environment setup

In [2]:
# Resolve current working directory
CWD = Path.cwd()
print(f'Current working directory: {CWD}')


# --- Set data path based on environment

# If we are running on kaggle...
if CWD.parts[1] == 'kaggle':
    print('Kaggle environment')
    data_path = '/kaggle/input/datasets/gperdrizet/rsna-knee-abnormality-pseudolabeled-training-data'

    # Check that the dataset is present
    if Path(data_path).is_dir() == False:
        raise FileNotFoundError(f'Dataset directory not found, add as input.')
        

# If we are somewhere else...
else:
    import kagglehub
    print('Non-kaggle environment')
    data_path = kagglehub.dataset_download(
        'gperdrizet/rsna-knee-abnormality-pseudolabeled-training-data',
        force_download=True
    )

print(f'Data path: {data_path}')

Current working directory: /workspaces/RSNA-knee-abnormality/notebooks
Non-kaggle environment


100%|██████████| 2.06M/2.06M [00:00<00:00, 6.66MB/s]

Extracting files...
Data path: /home/vscode/.cache/kagglehub/datasets/gperdrizet/rsna-knee-abnormality-pseudolabeled-training-data/versions/7


## 2. Data loading

In [3]:
# Load to dataframe
df = pd.read_csv(f'{data_path}/pseudolabeled-train.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4405 entries, 0 to 4404
Data columns (total 39 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   StudyInstanceUID            4405 non-null   object 
 1   Report                      4405 non-null   object 
 2   ACL                         4405 non-null   int64  
 3   MCL                         4405 non-null   int64  
 4   Medial Meniscus             4405 non-null   int64  
 5   Lateral Meniscus            4405 non-null   int64  
 6   Medial OA                   4405 non-null   int64  
 7   Lateral OA                  4405 non-null   int64  
 8   PF OA                       4405 non-null   int64  
 9   Effusion                    4405 non-null   int64  
 10  Synovitis                   4405 non-null   int64  
 11  Baker's                     4405 non-null   int64  
 12  Contusion                   4405 non-null   int64  
 13  Fracture                    4405 

## 3. Abnormality distributions

In [7]:
abnormalities = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

In [15]:
melted_df = df.melt(
    id_vars=['label_source'],
    value_vars=abnormalities, 
    var_name="Abnormality",
    value_name="Call"
)

melted_df.head()

,label_source,Abnormality,Call
0,gold,ACL,0
1,gold,ACL,0
2,gold,ACL,0
3,gold,ACL,0
4,gold,ACL,1


In [16]:
contingency_table = pd.crosstab(
    index=[melted_df["Abnormality"], melted_df["Call"]], 
    columns=melted_df["label_source"],
    margins=True  # Optional: Adds row/column totals
)

print(contingency_table)

label_source           gold  pseudo    All
Abnormality      Call                     
ACL              0       34    3614   3648
                 1       24     733    757
Baker's          0       46    3327   3373
                 1       12    1020   1032
Contusion        0       39    3889   3928
                 1       19     458    477
Effusion         0       23    2120   2143
                 1       35    2227   2262
Fracture         0       40    4097   4137
                 1       18     250    268
Lateral Meniscus 0       35    3578   3613
                 1       23     769    792
Lateral OA       0       47    3713   3760
                 1       11     634    645
MCL              0       49    3949   3998
                 1        9     398    407
Medial Meniscus  0       32    2389   2421
                 1       26    1958   1984
Medial OA        0       43    3049   3092
                 1       15    1298   1313
PF OA            0       37    2737   2774
           

In [17]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [19]:
results = []

for feat in abnormalities:
    # 1. Build the 2x2 contingency table
    ct = pd.crosstab(df['label_source'], df[feat])

    # 2. Run Fisher's Exact Test uniformly
    # odds_ratio measures the strength/direction of the effect
    odds_ratio, p_val = fisher_exact(ct)
    
    results.append({
        "Feature": feat,
        "Odds Ratio": odds_ratio,
        "Raw p-value": p_val
    })

results_df = pd.DataFrame(results)
display(results_df)

,Feature,Odds Ratio,Raw p-value
0,ACL,0.287332,1.097711e-05
1,MCL,0.548718,1.067803e-01
2,Medial Meniscus,1.008726,1.000000e+00
3,Lateral Meniscus,0.327059,8.901765e-05
4,Medial OA,1.220378,5.656799e-01
5,Lateral OA,0.729574,3.489539e-01
6,PF OA,1.036415,1.000000e+00
7,Effusion,0.690310,1.869099e-01
8,Synovitis,0.291840,8.782181e-06
9,Baker's,1.175233,7.550943e-01


In [21]:
rejected, corrected_pvals, _, _ = multipletests(
    results_df["Raw p-value"], alpha=0.05, method="fdr_bh"
)

results_df["FDR Corrected p-value"] = corrected_pvals
results_df["Significant (α=0.05)"] = rejected

display(results_df.sort_values(by="FDR Corrected p-value"))

,Feature,Odds Ratio,Raw p-value,FDR Corrected p-value,Significant (α=0.05)
11,Fracture,0.135601,3.674408e-09,4.409289e-08,True
10,Contusion,0.241734,5.327681e-06,3.196608e-05,True
0,ACL,0.287332,1.097711e-05,3.293133e-05,True
8,Synovitis,0.291840,8.782181e-06,3.293133e-05,True
3,Lateral Meniscus,0.327059,8.901765e-05,2.136424e-04,True
1,MCL,0.548718,1.067803e-01,2.135606e-01,False
7,Effusion,0.690310,1.869099e-01,3.204170e-01,False
5,Lateral OA,0.729574,3.489539e-01,5.234309e-01,False
4,Medial OA,1.220378,5.656799e-01,7.542399e-01,False
9,Baker's,1.175233,7.550943e-01,9.061131e-01,False
